In [1]:
import duckdb
from networks.feed_forward_more_features import NeuralModel
from networks.cnn_network import build_dataloaders

con = duckdb.connect('../capillary.db')

df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


print(f"Antal fall med M-komponent:{(df['label'] == 1).sum()}")
print(f"Antal fall utan M-komponent:{(df['label'] == 0).sum()}")


train_rows = df[df['set'] == 'train']
val_rows = df[df['set'] == 'val']
test_rows = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index


train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

network = NeuralModel()
network.reset_weights()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
network.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal fall med M-komponent:2942
Antal fall utan M-komponent:69882
Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 256,578
Epoch   0 | train: 0.9401 | val: 0.4332 | acc: 94.43% | AUC: 0.789  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features.pth
Epoch   1 | train: 0.5573 | val: 0.3813 | acc: 94.40% | AUC: 0.844  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features.pth
Epoch   2 | train: 0.5017 | val: 0.5780 | acc: 68.60% | AUC: 0.835  | LR: 0.001
Epoch   3 | train: 0.4882 | val: 0.5211 | acc: 82.14% | AUC: 0.884  | LR: 0.001
Epoch   4 | train: 0.4588 | val: 0.4046 | acc: 93.55% | AUC: 0.860  | LR: 0.001
Epoch   5 | train: 0.4340 | val: 0.2664 | acc: 96.08% | AUC: 0.919  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features.pth
Epoch   6 | train: 0.4083 | val: 0.3681 | acc: 89.81% | AUC: 0.915  | LR: 0.001
Epoch   7 | train: 0.3827 | val: 0.3

In [2]:
con = duckdb.connect('../capillary.db')

df = con.execute(""" SELECT row_id,fractions, boundaries, value, label, albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm,set FROM protein_data WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL""").df()

con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


network = NeuralModel()
network.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 256,578
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
Epoch   0 | train: 0.8839 | val: 0.6240 | acc: 90.24% | AUC: 0.751  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_fold1.pth
Epoch   1 | train: 0.5529 | val: 0.5592 | acc: 91.54% | AUC: 0.815  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_fold1.pth
Epoch   2 | train: 0.5048 | val: 0.5027 | acc: 91.50% | AUC: 0.804  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_fold1.pth
Epoch   3 | train: 0.4523 | val: 0.4637 | acc: 89.73% | AUC: 0.832  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_fold1.pth
Epoch   4 | train: 0.4904 | val: 0.4508 | acc: 87.36% | AUC: 0.850  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_fold1.pth
Epoch   5 | train

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.124109,0.216273,94.470260,0.966926,0.955390,0.869888,1799,84,35,234
1,2,0.149159,0.222401,94.939647,0.955248,0.960743,0.869888,1811,74,35,234
2,3,0.158274,0.234637,94.842007,0.958212,0.960701,0.862454,1809,74,37,232
3,4,0.145622,0.237448,94.426382,0.962469,0.957006,0.855019,1803,81,39,230
4,5,0.159408,0.249355,94.475395,0.955745,0.957029,0.858736,1804,81,38,231
5,6,0.183046,0.217519,94.379935,0.966501,0.958068,0.843866,1805,79,42,227
6,7,0.179946,0.233677,94.702602,0.962488,0.964418,0.825279,1816,67,47,222
7,8,0.157007,0.214553,93.964717,0.966856,0.948011,0.881041,1787,98,32,237
8,9,0.115139,0.183113,95.961003,0.972030,0.968153,0.900000,1824,60,27,243
9,10,0.146859,0.166855,95.401765,0.977447,0.960701,0.907407,1809,74,25,245


In [3]:
from functions.evaluation import evaluate


test_rows = test_rows[test_rows['label'].isin([0,1])]
test_rows['cnn_probability'] = test_rows['probability'].copy()
result = network.predict(test_rows)
_ = evaluate(result)

KeyError: 'probability'